In [ ]:
!python -V

Python 3.12.13


# 1.Carga dos Dados

Nesta etapa inicial, realizamos o carregamento das bases de dados referentes às despesas públicas das cidades pertencentes à Região Metropolitana de João Pessoa, incluindo:
João Pessoa
Bayeux
Santa Rita
Cabedelo
Conde
Alhandra
Pitimbu
Pedras de Fogo
Rio Tinto
Lucena

Os dados foram coletados a partir do portal oficial de dados abertos do Tribunal de Contas do Estado da Paraíba, disponível em:

https://dados-abertos.tce.pb.gov.br/

O objetivo desta etapa é construir uma visão inicial sobre:

os gastos realizados por cada município;
a distribuição orçamentária entre diferentes áreas administrativas;
os padrões de aplicação de recursos públicos;
possíveis diferenças entre os municípios analisados.


In [4]:
import pandas as pd
import numpy as np
import requests

# Lista de arquivos (cidade, ano, url)
dados = [
    ('santa_rita', 2025, 'https://raw.githubusercontent.com/Andersonlima13/ML_TOPICOS/refs/heads/main/despesas-2025-santarita.csv'),
    ('santa_rita', 2026, 'https://raw.githubusercontent.com/Andersonlima13/ML_TOPICOS/refs/heads/main/despesas-2026-santarita.csv'),

    ('alhandra', 2025, 'https://raw.githubusercontent.com/Andersonlima13/ML_TOPICOS/refs/heads/main/despesas-2025-alhandra.csv'),
    ('alhandra', 2026, 'https://raw.githubusercontent.com/Andersonlima13/ML_TOPICOS/refs/heads/main/despesas-2026-alhandra.csv'),

    ('cruz_do_espirito_santo', 2025, 'https://raw.githubusercontent.com/Andersonlima13/ML_TOPICOS/refs/heads/main/despesas-2025-cruzdoespiritosanto.csv'),
    ('cruz_do_espirito_santo', 2026, 'https://raw.githubusercontent.com/Andersonlima13/ML_TOPICOS/refs/heads/main/despesas-2026-cruzdoespiritosanto.csv'),

    ('pitimbu', 2025, 'https://raw.githubusercontent.com/Andersonlima13/ML_TOPICOS/refs/heads/main/despesas-2025-pitimbu.csv'),
    ('pitimbu', 2026, 'https://raw.githubusercontent.com/Andersonlima13/ML_TOPICOS/refs/heads/main/despesas-2026-pitimbu.csv'),


    ('lucena', 2026, 'https://raw.githubusercontent.com/Andersonlima13/ML_TOPICOS/refs/heads/main/despesas-2026-lucena.csv'),
    ('lucena', 2025, 'https://raw.githubusercontent.com/Andersonlima13/ML_TOPICOS/refs/heads/main/despesas-2025-lucena.csv'),

    ('bayeux', 2025, 'https://raw.githubusercontent.com/Andersonlima13/ML_TOPICOS/refs/heads/main/despesas-2025-bayeux.csv'),
    ('bayeux', 2026, 'https://raw.githubusercontent.com/Andersonlima13/ML_TOPICOS/refs/heads/main/despesas-2026-bayeux.csv'),

    ('joao_pessoa', 2026, 'https://raw.githubusercontent.com/Andersonlima13/ML_TOPICOS/refs/heads/main/despesas-2026-joaopessoa.csv'),
    ('joao_pessoa', 2025, 'https://raw.githubusercontent.com/Andersonlima13/ML_TOPICOS/refs/heads/main/despesas-2025-joaopessoa.csv'),

    ('caapora', 2025, 'https://raw.githubusercontent.com/Andersonlima13/ML_TOPICOS/refs/heads/main/despesas-2025-caapora.csv'),
    ('caapora', 2026, 'https://raw.githubusercontent.com/Andersonlima13/ML_TOPICOS/refs/heads/main/despesas-2026-caapora.csv'),

    ('cabedelo', 2025, 'https://raw.githubusercontent.com/Andersonlima13/ML_TOPICOS/refs/heads/main/despesas-2025-cabedelo.csv'),
    ('cabedelo', 2026, 'https://raw.githubusercontent.com/Andersonlima13/ML_TOPICOS/refs/heads/main/despesas-2026-cabedelo.csv'),

    ('conde', 2025, 'https://raw.githubusercontent.com/Andersonlima13/ML_TOPICOS/refs/heads/main/despesas-2025-conde.csv'),
    ('conde', 2026, 'https://raw.githubusercontent.com/Andersonlima13/ML_TOPICOS/refs/heads/main/despesas-2026-conde.csv'),

    ('pedras_de_fogo', 2025, 'https://raw.githubusercontent.com/Andersonlima13/ML_TOPICOS/refs/heads/main/despesas-2025-pedrasdefogo.csv'),
    ('pedras_de_fogo', 2026, 'https://raw.githubusercontent.com/Andersonlima13/ML_TOPICOS/refs/heads/main/despesas-2026-pedrasdefogo.csv'),

    ('rio_tinto', 2025, 'https://raw.githubusercontent.com/Andersonlima13/ML_TOPICOS/refs/heads/main/despesas-2025-riotinto.csv'),
    ('rio_tinto', 2026, 'https://raw.githubusercontent.com/Andersonlima13/ML_TOPICOS/refs/heads/main/despesas-2026-riotinto.csv'),
]

nomes = {
    'joao_pessoa':    'João Pessoa',
    'pedras_de_fogo': 'Pedras de Fogo',
    'rio_tinto':      'Rio Tinto',
    'santa_rita':     'Santa Rita',
    'caapora':        'Caaporã',
    'cabedelo':       'Cabedelo',
    'alhandra':       'Alhandra',
    'bayeux':         'Bayeux',
    'conde':          'Conde',
    'lucena':         'Lucena',
    'pitimbu':        'Pitimbu',
    'cruz_do_espirito_santo': 'Cruz do Espirito Santo',

}

def detectar_sep(url):
    sample = requests.get(url).text[:2000]
    return ',' if sample.count(',') > sample.count(';') else ';'

dfs = []

for cidade, ano, url in dados:
    try:
        sep = detectar_sep(url)

        df = pd.read_csv(
            url,
            sep=sep,
            encoding='utf-8-sig',
            decimal=',',
            thousands='.',
            quotechar='"',
            on_bad_lines='warn'
        )

        df['cidade'] = cidade
        df['ano'] = ano

        # garante coluna municipio preenchida com nome correto
        nome_correto = nomes.get(cidade, cidade.replace('_', ' ').title())
        if 'municipio' not in df.columns:
            df['municipio'] = nome_correto
        else:
            df['municipio'] = df['municipio'].fillna(nome_correto)

        dfs.append(df)
        print(f"Carregado: {cidade} {ano}")

    except Exception as e:
        print(f"Erro em {cidade} {ano}: {e}")

df_total = pd.concat(dfs, ignore_index=True)

df_total.head()

Carregado: santa_rita 2025
Carregado: santa_rita 2026
Carregado: alhandra 2025
Carregado: alhandra 2026
Carregado: cruz_do_espirito_santo 2025
Carregado: cruz_do_espirito_santo 2026
Carregado: pitimbu 2025
Carregado: pitimbu 2026
Carregado: lucena 2026
Carregado: lucena 2025
Carregado: bayeux 2025
Carregado: bayeux 2026
Carregado: joao_pessoa 2026
Carregado: joao_pessoa 2025
Carregado: caapora 2025
Carregado: caapora 2026
Carregado: cabedelo 2025
Carregado: cabedelo 2026
Carregado: conde 2025
Carregado: conde 2026
Carregado: pedras_de_fogo 2025
Carregado: pedras_de_fogo 2026
Carregado: rio_tinto 2025
Carregado: rio_tinto 2026


,municipio,codigo_unidade_gestora,descricao_unidade_gestora,numero_empenho,data_empenho,mes,cpf_cnpj,nome_credor,valor_empenhado,valor_liquidado,...,modalidade_licitacao,numero_obra,historico,codigo_fonte_recurso,descricao_fonte_recurso,ano_fonte,co,descricao_co,cidade,ano
0,Santa Rita,602171.0,Fundo Municipal de Saúde de Santa Rita,4,2025-01-06,01-Janeiro,9433715000102,FUNDACAO GOV.FLAVIO RIBEIRO COUTINHO,765.00,765.00,...,Sem Licitação,0,Referente a Administração de Contraste e sedaç...,500.0,Recursos não vinculados de Impostos,1,1002.0,Identificação das despesas com ações e serviço...,santa_rita,2025
1,Santa Rita,602171.0,Fundo Municipal de Saúde de Santa Rita,2,2025-01-06,01-Janeiro,9433715000102,FUNDACAO GOV.FLAVIO RIBEIRO COUTINHO,234313.93,234313.93,...,Sem Licitação,0,Valor Referente Repasse financeiro IAC.Constan...,600.0,Transferências Fundo a Fundo de Recursos do SU...,1,NaN,NaN,santa_rita,2025
2,Santa Rita,602171.0,Fundo Municipal de Saúde de Santa Rita,1,2025-01-06,01-Janeiro,9095183000140,ENERGISA PARAIBA - Distribuidora de Energia S/A,1599.34,1599.34,...,Sem Licitação,0,Valor que se empenha referente a Conta energia...,600.0,Transferências Fundo a Fundo de Recursos do SU...,1,NaN,NaN,santa_rita,2025
3,Santa Rita,602171.0,Fundo Municipal de Saúde de Santa Rita,3,2025-01-06,01-Janeiro,9433715000102,FUNDACAO GOV.FLAVIO RIBEIRO COUTINHO,29936.73,29936.73,...,Sem Licitação,0,Referente a Administração de Contraste e sedaç...,600.0,Transferências Fundo a Fundo de Recursos do SU...,1,NaN,NaN,santa_rita,2025
4,Santa Rita,601171.0,Fundo Municipal de Assistência Social de Santa...,16,2025-01-14,01-Janeiro,34681704000199,AGUAS DO NORDESTE S.A,0.00,0.00,...,Sem Licitação,0,VALOR QUE SE EMPENHA REFERENTE AO SERVIÇO DE F...,500.0,Recursos não vinculados de Impostos,1,NaN,NaN,santa_rita,2025


# 1.2 Adcionando coluna de "Ano"

Para facilitar a segmentação temporal da base de dados, utilizamos o campo ano_referencia como identificador do ano-base de cada registro.

In [5]:
# adiciona coluna ano_referencia baseada na coluna 'ano' já existente
df_total['ano_referencia'] = df_total['ano'].astype(int)

# separa em dois dataframes por ano
df_2025 = df_total[df_total['ano_referencia'] == 2025].copy()
df_2026 = df_total[df_total['ano_referencia'] == 2026].copy()

print(f"Total geral:  {len(df_total):,} registros")
print(f"Ano 2025:     {len(df_2025):,} registros")
print(f"Ano 2026:     {len(df_2026):,} registros")

# confirma distribuição por cidade e ano
display(
    df_total.groupby(['municipio', 'ano_referencia'])
    .size()
    .reset_index(name='registros')
    .sort_values(['municipio', 'ano_referencia'])
)

Total geral:  216,767 registros
Ano 2025:     169,389 registros
Ano 2026:     47,378 registros


,municipio,ano_referencia,registros
0,Alhandra,2025,13279
1,Alhandra,2026,2481
2,Bayeux,2025,9618
3,Bayeux,2026,2528
4,Caaporã,2025,11987
5,Caaporã,2026,3088
6,Cabedelo,2025,9696
7,Cabedelo,2026,2653
8,Conde,2025,11767
9,Conde,2026,2582


#2.Explorando o dataset



O objetivo de obter um panorama geral dados em relação a:
1. Conhecer seus metadados
2. Explorar o domínio de valores de seus atributos
3. Identificar a existências de valores nulos, entradas de dados incorretas e formato dos dados
4. Levantar estatísticas sobre os dados.


##2.1 Mapeando os campos unicos de função para cada cidade

Nesta etapa, buscamos identificar e mapear os valores únicos presentes no campo de função dentro dos datasets de cada município.

O objetivo é compreender:

    -como os gastos públicos estão organizados administrativamente;

    -quais áreas recebem maior destinação de recursos;

    -como cada município classifica suas despesas.


Entre os exemplos de funções encontradas estão áreas como:

Saúde,
Educação,
Urbanismo,
Segurança,
Legislativo,
Administração,

In [6]:
map_funcao = (
    df_total[['codigo_funcao', 'funcao']]
    .drop_duplicates()
    .sort_values('codigo_funcao')
    .reset_index(drop=True)
)

print(map_funcao.to_string(index=False))

 codigo_funcao                funcao
             1           Legislativa
             2            Judiciária
             3   Essencial à Justica
             4         Administração
             6     Segurança Pública
             8   Assistêncial Social
             9    Previdência Social
            10                 Saúde
            11              Trabalho
            12              Educação
            13               Cultura
            14 Direitos de Cidadania
            15             Urbanismo
            16             Habitação
            17            Saneamento
            18      Gestão Ambiental
            19  Ciência e Tecnologia
            20           Agricultura
            22             Indústria
            23   Comércio e Serviços
            24          Comunicações
            25               Energia
            26            Transporte
            27      Desporto e Lazer
            28     Encargos Especias


##2.2 Mapeando unicidade dos campos de açao.
Nesta etapa, exploramos os valores únicos do campo de ação, responsável por detalhar de forma mais específica o destino de cada gasto público.

O objetivo principal é identificar:

    -quais programas e iniciativas recebem recursos;
    como os municípios executam suas despesas;
    quais atividades possuem maior recorrência orçamentária.

In [7]:
map_acao = (
    df_total[['codigo_acao', 'acao' , 'municipio']]
    .drop_duplicates()
    .sort_values('codigo_acao')
    .reset_index(drop=True)
)

print(map_acao.to_string(index=False))

 codigo_acao                                                                   acao              municipio
           1                   AMORTIZAÇÃO DA DÍVIDA CONTATADA DO PODER LEGISLATIVO               Alhandra
           1                                    AMORTIZACAO DA DIVIDA - PRECATORIOS Cruz do Espírito Santo
           1                                     AMORTIZAÇÃO DE DÍVIDAS CONTRATADAS                  Conde
           1                  AMORTIZAR DIVIDAS - SENTENCAS JUDICIAIS (PRECATORIOS)               Cabedelo
           1                        PAGAMENTO DE PRECATORIOS JUDICIAIS DO MUNICIPIO              Rio Tinto
           1 AMORTIZAÇÃO DE DÍVIDAS CONTRATUAIS (INSS, FGTS, ENERGISA, CAGEPA, IBAM                 Lucena
           2                           AMORTIZACAO DA DIVIDA FUNDADA PREVIDENCIARIA Cruz do Espírito Santo
           2                          ADMINISTRAÇÃO E COORDENAÇÃO DA DÍVIDA INTERNA             Santa Rita
           2                         

##2.3 Mapeando os campos de subfunção

Dando continuidade à segmentação dos dados, exploramos os campos de subfunção, responsáveis por detalhar ainda mais as categorias administrativas presentes na função principal.

Enquanto a função representa uma área ampla da administração pública, a subfunção fornece um nível adicional de detalhamento, nos dando maior precisão em :

    -onde os recursos estão sendo aplicados;
    quais atividades específicas recebem investimento;
    como os gastos estão distribuídos internamente em cada setor

In [8]:
map_subfuncao = (
    df_total[['codigo_subfuncao', 'subfuncao']]
    .drop_duplicates()
    .sort_values('subfuncao')
    .reset_index(drop=True)
)

print(map_subfuncao.to_string(index=False))

 codigo_subfuncao                                          subfuncao
              605                                      Abastecimento
              123                           Administração Financeira
              122                                Administração Geral
              129                          Administração de Receitas
              306                             Alimentação e Nutrição
              244                            Assistência Comunitária
              302              Assistência Hospitalar e Ambulatorial
              241                               Assistência ao Idoso
              242             Assistência ao Portador de Deficiência
              423                    Assistência aos Povos Indígenas
              243             Assistência à Criança e ao Adolescente
              301                                     Atenção Básica
               61                                    Ação Judiciária
               31                 

##2.3 Mapeando por unicidade os campos de unidade orçamentaria:

In [9]:
map_unidade = (
    df_total[['municipio', 'codigo_unidade_orcamentaria', 'descricao_unidade_orcamentaria']]
    .drop_duplicates()
    .sort_values('codigo_unidade_orcamentaria')
    .reset_index(drop=True)
)

print(map_unidade.to_string(index=False))

             municipio  codigo_unidade_orcamentaria                     descricao_unidade_orcamentaria
        Pedras de Fogo                          101                 CAMARA MUNICIPAL DE PEDRAS DE FOGO
        Pedras de Fogo                          201                    SECRETARIA MUNICIPAL DE GOVERNO
        Pedras de Fogo                          202              SECRETARIA MUNICIPAL DE ADMINISTRAÇÃO
        Pedras de Fogo                          203    SECRETARIA MUNICIPAL DE FINANÇAS E PLANEJAMENTO
        Pedras de Fogo                          204 SECRETARIA MUNICIPAL DE EDUCAÇÃO, CULTURA E DESPOR
        Pedras de Fogo                          205 SECRETARIA MUNICIPAL  DE DESENVOLVIMENTO ECONÔMICO
        Pedras de Fogo                          205 SECRETARIA MUNICIPAL DE AGRICULTURA E DESENVOLVIME
        Pedras de Fogo                          206                      SECRETARIA MUNICIPAL DE SAÚDE
        Pedras de Fogo                          207 SECRETARIA MUNICIPAL 

#3. Filtro base

Inicialmente, aplicamos um filtro-base com o objetivo de restringir os dados apenas aos registros relacionados à:

Câmara Municipal;
Poder Legislativo.


Esse isolamento é importante para separar despesas legislativas dos demais gastos administrativos do município, permitindo análises mais específicas sobre:

    -custos operacionais da câmara;
    despesas com pessoal;
    distribuição de recursos legislativos;
    padrões de gasto do poder legislativo municipal.


In [10]:
df_leg = df_total[
    (df_total['codigo_funcao'] == 1) &
    (
        df_total['descricao_unidade_orcamentaria']
        .astype(str)
        .str.upper()
        .str.normalize('NFKD')
        .str.encode('ascii', errors='ignore')
        .str.decode('utf-8')
        .str.contains('CAMARA|LEGISLATIVO', na=False)
    )
]

print(f"Registros da Câmara: {df_leg.shape[0]}")

df_leg[['municipio', 'codigo_funcao', 'codigo_unidade_orcamentaria', 'descricao_unidade_orcamentaria']].head()

Registros da Câmara: 11375


,municipio,codigo_funcao,codigo_unidade_orcamentaria,descricao_unidade_orcamentaria
31,Santa Rita,1,1010,CÂMARA MUNICIPAL
32,Santa Rita,1,1010,CÂMARA MUNICIPAL
33,Santa Rita,1,1010,CÂMARA MUNICIPAL
34,Santa Rita,1,1010,CÂMARA MUNICIPAL
35,Santa Rita,1,1010,CÂMARA MUNICIPAL


#3.1. isolar GASTOS COM PESSOAL

Nesta etapa, iniciamos o processo de isolamento das despesas relacionadas a gastos com pessoal.

O objetivo é identificar despesas vinculadas a:

salários;
folha de pagamento;
encargos sociais;
previdência;
obrigações patronais;
subsídios e remunerações.

Antes da aplicação dos filtros, realizamos a conversão do campo valor_pago para formato numérico, garantindo maior precisão nas operações de agregação, soma e análise financeira.

In [11]:
df_total['valor_pago'] = (
    df_total['valor_pago']
    .astype(str)
    .str.replace('.', '', regex=False)
    .str.replace(',', '.', regex=False)
)

df_total['valor_pago'] = pd.to_numeric(df_total['valor_pago'], errors='coerce')

df_total['municipio'] = (
    df_total['municipio']
    .astype(str)
    .str.strip()
    .str.upper()
)

#3.2. isolando os gastos com encargo e pessoal

Após a investigação inicial dos dados, realizando o isolamento das despesas relacionadas à Câmara Municipal e ao Poder Legislativo, foi definido um conjunto de padrões textuais para identificar registros associados a gastos com pessoal, remuneração e encargos sociais.

Os filtros utilizados consideram termos relacionados a salários, benefícios, previdência e obrigações patronais, permitindo segmentar apenas despesas vinculadas à folha de pagamento e custos trabalhistas.

Os padrões definidos foram:

"PESSOAL|ENCARGOS SOCIAIS|PREVIDENCI|"
r"REMUNERACAO|SUBSIDIO|VENCIMENTOS|FOLHA|"
r"OBRIGACOES PATRONAIS|LEGISLATIVO"

In [12]:
map_acao['acao_clean'] = (
    map_acao['acao']
    .astype(str)
    .str.upper()
    .str.normalize('NFKD')
    .str.encode('ascii', errors='ignore')
    .str.decode('utf-8')
)

codigos_pessoal = map_acao[
    map_acao['acao_clean'].str.contains(
        r'PESSOAL|ENCARGOS SOCIAIS|PREVIDENCI|'
        r'REMUNERACAO|SUBSIDIO|VENCIMENTOS|FOLHA|'
        r'OBRIGACOES PATRONAIS|LEGISLATIVO',
        na=False
    )
][['municipio', 'codigo_acao', 'acao']] \
.drop_duplicates() \
.sort_values(['municipio', 'codigo_acao']) \
.reset_index(drop=True)

# exibição
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', 100)

display(codigos_pessoal)

,municipio,codigo_acao,acao
0,Alhandra,1,AMORTIZAÇÃO DA DÍVIDA CONTATADA DO PODER LEGISLATIVO
1,Alhandra,2002,MANUTENÇÃO DOS ENCARGOS PREVIDENCIÁIRIOS DA CÂMARA MUNICIPAL
2,Alhandra,2044,MANUTENÇÃO DOS BENEFÍCIOS PREVIDENCIÁRIOS DOS SERVIDORES MUNICIPAIS
3,Alhandra,2171,BENEFÍCIOS PREVIDENCIARIOS DOS SERVIDORES MUNICIPAIS
4,Bayeux,2001,MANUTENCAO DAS ATIVIDADES DO PODER LEGISLATIVO MUNICIPAL
5,Bayeux,2113,ADMINISTRACAO DO INSTITUTO DE PREVIDENCIA DOS SERVIDORES MUNICIPAIS -
6,Bayeux,2114,CONCEBER BENEFICIOS AOS SEGURADOS DO PODER LEGISLATIVO
7,Cabedelo,2001,"Manutencao, Modernizacao e Desenvolvimento do Poder Legislativo"
8,Cabedelo,2001,MANUTENCAO DAS ATIVIDADES FINS DO PODER LEGISLATIVO
9,Cabedelo,2032,Conceder Beneficios Previdenciarios


#3.3. Isolando Campos Já Explorados — Gastos com Pessoal


Após a definição inicial dos padrões relacionados a gastos com pessoal, realizamos uma nova etapa de segmentação com o objetivo de refinar ainda mais os dados analisados.

Durante a exploração da base, foi identificado que alguns registros classificados inicialmente como despesas pessoais incluíam gastos que não estavam diretamente ligados à manutenção operacional do Poder Legislativo.


aplicamos filtros adicionais para remover:

    -despesas administrativas não vinculadas à folha legislativa;
    registros genéricos sem relação direta com manutenção do legislativo;
    gastos classificados de forma ampla, mas sem vínculo direto com pessoal da câmara.

In [13]:
def normalizar(s):
    return (
        s.astype(str)
        .str.strip()
        .str.upper()
        .str.normalize('NFKD')
        .str.encode('ascii', errors='ignore')
        .str.decode('utf-8')
    )

# blacklist — codigos que NAO pertencem a camara/legislativo
blacklist = [
    #('ALHANDRA', 2044),
    #('ALHANDRA', 2171),
    ('BAYEUX', 2113),
    ('JOAO PESSOA', 11),
    ('JOAO PESSOA', 19),
    ('JOAO PESSOA', 32),
    ('JOAO PESSOA', 67),
    ('JOAO PESSOA', 164),
    ('JOAO PESSOA', 328),
    ('JOAO PESSOA', 330),
    ('JOAO PESSOA', 361),
    ('JOAO PESSOA', 421),
    ('JOAO PESSOA', 462),
    ('JOAO PESSOA', 463),
    ('JOAO PESSOA', 583),
    ('JOAO PESSOA', 604),
    ('JOAO PESSOA', 610),
    ('JOAO PESSOA', 634),
    ('JOAO PESSOA', 708),
    ('JOAO PESSOA', 709),
    ('JOAO PESSOA', 710),
    ('JOAO PESSOA', 729),
    ('JOAO PESSOA', 730),
    ('JOAO PESSOA', 732),
    ('JOAO PESSOA', 767),
    ('JOAO PESSOA', 838),
    ('JOAO PESSOA', 907),
    ('PEDRAS DE FOGO', 2007),
    ('PEDRAS DE FOGO', 2192),
    ('RIO TINTO', 4),
    ('SANTA RITA', 1092),
]

blacklist_df = pd.DataFrame(blacklist, columns=['municipio', 'codigo_acao'])
blacklist_df['municipio'] = normalizar(blacklist_df['municipio'])

codigos_pessoal['municipio_norm'] = normalizar(codigos_pessoal['municipio'])

codigos_pessoal_filtrado = (
    codigos_pessoal
    .merge(
        blacklist_df.rename(columns={'municipio': 'municipio_norm'}),
        on=['municipio_norm', 'codigo_acao'],
        how='left',
        indicator=True
    )
    .query("_merge == 'left_only'")
    .drop(columns=['_merge', 'municipio_norm'])
    .reset_index(drop=True)
)

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', 100)

display(codigos_pessoal_filtrado)

,municipio,codigo_acao,acao
0,Alhandra,1,AMORTIZAÇÃO DA DÍVIDA CONTATADA DO PODER LEGISLATIVO
1,Alhandra,2002,MANUTENÇÃO DOS ENCARGOS PREVIDENCIÁIRIOS DA CÂMARA MUNICIPAL
2,Alhandra,2044,MANUTENÇÃO DOS BENEFÍCIOS PREVIDENCIÁRIOS DOS SERVIDORES MUNICIPAIS
3,Alhandra,2171,BENEFÍCIOS PREVIDENCIARIOS DOS SERVIDORES MUNICIPAIS
4,Bayeux,2001,MANUTENCAO DAS ATIVIDADES DO PODER LEGISLATIVO MUNICIPAL
5,Bayeux,2114,CONCEBER BENEFICIOS AOS SEGURADOS DO PODER LEGISLATIVO
6,Cabedelo,2001,"Manutencao, Modernizacao e Desenvolvimento do Poder Legislativo"
7,Cabedelo,2001,MANUTENCAO DAS ATIVIDADES FINS DO PODER LEGISLATIVO
8,Cabedelo,2032,Conceder Beneficios Previdenciarios
9,Cabedelo,2033,Manter as Atividades Previdenciarias


#3.4. Mapeando os campos antes de verificar os gastos com pessoal

Antes de iniciar a análise dos gastos com pessoal, realizamos uma etapa de exploração e mapeamento estrutural dos dados presentes na base.

O objetivo desta fase é compreender:

    -os tipos de dados presentes em cada coluna;
    possíveis inconsistências de formatação;
    valores nulos ou ausentes;
    padrões textuais existentes;
    estrutura geral dos registros financeiros.

In [14]:
print("=== SHAPE ===")
print(f"df_leg: {df_leg.shape}")
print(f"map_acao: {map_acao.shape}")

print("\n=== COLUNAS df_leg ===")
print(df_leg.columns.tolist())

print("\n=== COLUNAS map_acao ===")
print(map_acao.columns.tolist())

print("\n=== DTYPES df_leg ===")
print(df_leg.dtypes)

print("\n=== SAMPLE df_leg — municipio + codigo_acao + valor_pago ===")
print(df_leg[['municipio', 'codigo_acao', 'valor_pago']].head(10).to_string())

print("\n=== SAMPLE codigos_pessoal — municipio + codigo_acao ===")
print(codigos_pessoal[['municipio', 'codigo_acao']].head(10).to_string())

print("\n=== NULOS valor_pago ===")
print(f"nulos: {df_leg['valor_pago'].isna().sum()} / {len(df_leg)}")

print("\n=== TIPO municipio ===")
print(f"df_leg:        {df_leg['municipio'].dtype} | exemplo: {df_leg['municipio'].iloc[0]!r}")
print(f"codigos_pessoal: {codigos_pessoal['municipio'].dtype} | exemplo: {codigos_pessoal['municipio'].iloc[0]!r}")

print("\n=== TIPO codigo_acao ===")
print(f"df_leg:        {df_leg['codigo_acao'].dtype} | exemplo: {df_leg['codigo_acao'].iloc[0]!r}")
print(f"codigos_pessoal: {codigos_pessoal['codigo_acao'].dtype} | exemplo: {codigos_pessoal['codigo_acao'].iloc[0]!r}")

=== SHAPE ===
df_leg: (11375, 43)
map_acao: (2106, 4)

=== COLUNAS df_leg ===
['municipio', 'codigo_unidade_gestora', 'descricao_unidade_gestora', 'numero_empenho', 'data_empenho', 'mes', 'cpf_cnpj', 'nome_credor', 'valor_empenhado', 'valor_liquidado', 'valor_pago', 'codigo_unidade_orcamentaria', 'descricao_unidade_orcamentaria', 'codigo_funcao', 'funcao', 'codigo_subfuncao', 'subfuncao', 'codigo_programa', 'programa', 'codigo_acao', 'acao', 'codigo_categoria_economica', 'categoria_economica', 'codigo_natureza', 'grupo_natureza_despesa', 'codigo_modalidade_aplicacao', 'modalidade_aplicacao', 'codigo_elemento_despesa', 'elemento_despesa', 'codigo_subelemento', 'codigo_subelemento_exibicao', 'numero_licitacao', 'modalidade_licitacao', 'numero_obra', 'historico', 'codigo_fonte_recurso', 'descricao_fonte_recurso', 'ano_fonte', 'co', 'descricao_co', 'cidade', 'ano', 'ano_referencia']

=== COLUNAS map_acao ===
['codigo_acao', 'acao', 'municipio', 'acao_clean']

=== DTYPES df_leg ===
municipi

#3.5. Filtrando despesas com o legislativo e valor gasto

Após a aplicação dos filtros definidos como despesas pessoais do Poder Legislativo, iniciamos o processo de segmentação e análise dos gastos identificados.

O objetivo desta etapa é investigar:

    -onde os gastos estão concentrados;
    como as despesas estão distribuídas;
    quais categorias possuem maior impacto financeiro;
    padrões de despesas relacionadas à manutenção do legislativo.




In [15]:
# garantir normalização consistente em TODOS os dataframes
def normalizar(s):
    return (
        s.astype(str)
        .str.strip()
        .str.upper()
        .str.normalize('NFKD')
        .str.encode('ascii', errors='ignore')
        .str.decode('utf-8')
    )

df_leg['municipio_norm'] = normalizar(df_leg['municipio'])
codigos_pessoal_filtrado['municipio_norm'] = normalizar(codigos_pessoal_filtrado['municipio'])


# 📌 1. identificar apenas códigos que realmente existem no df_leg
codigos_validos = (
    df_leg[['municipio_norm', 'codigo_acao']]
    .drop_duplicates()
)

codigos_pessoal_filtrado = (
    codigos_pessoal_filtrado
    .merge(
        codigos_validos,
        on=['municipio_norm', 'codigo_acao'],
        how='inner'
    )
    .reset_index(drop=True)
)


# 📌 2. anos disponíveis por município
anos_por_municipio = (
    df_leg[['municipio_norm', 'ano_referencia']]
    .drop_duplicates()
)


# 📌 3. expandir códigos para todos os anos disponíveis
codigos_pessoal_expandido = (
    codigos_pessoal_filtrado
    .merge(
        anos_por_municipio,
        on='municipio_norm',
        how='left'
    )
)


# 📌 4. agregar valores pagos
valores = (
    df_leg
    .groupby(['municipio_norm', 'codigo_acao', 'ano_referencia'])['valor_pago']
    .sum()
    .reset_index()
)


# 📌 5. merge final com valores
codigos_pessoal_valores = (
    codigos_pessoal_expandido
    .merge(
        valores,
        on=['municipio_norm', 'codigo_acao', 'ano_referencia'],
        how='left'
    )
)


# 📌 6. tratar tipos e valores
codigos_pessoal_valores['ano_referencia'] = (
    codigos_pessoal_valores['ano_referencia'].astype('Int64')
)

codigos_pessoal_valores['valor_pago'] = (
    codigos_pessoal_valores['valor_pago'].fillna(0)
)


# 📌 7. organizar resultado final
codigos_pessoal_valores = (
    codigos_pessoal_valores
    .sort_values(
        ['municipio', 'ano_referencia', 'valor_pago'],
        ascending=[True, True, False]
    )
    .reset_index(drop=True)
)


# 📌 8. configs de exibição
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', 200)
pd.set_option('display.float_format', '{:,.2f}'.format)


# 📌 9. exibir resultado
display(
    codigos_pessoal_valores[
        ['ano_referencia', 'municipio', 'codigo_acao', 'acao', 'valor_pago']
    ]
)

/tmp/ipykernel_14294/1036972465.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_leg['municipio_norm'] = normalizar(df_leg['municipio'])


,ano_referencia,municipio,codigo_acao,acao,valor_pago
0,2025,Alhandra,2002,MANUTENÇÃO DOS ENCARGOS PREVIDENCIÁIRIOS DA CÂMARA MUNICIPAL,"992,186.66"
1,2025,Alhandra,1,AMORTIZAÇÃO DA DÍVIDA CONTATADA DO PODER LEGISLATIVO,"19,886.81"
2,2026,Alhandra,2002,MANUTENÇÃO DOS ENCARGOS PREVIDENCIÁIRIOS DA CÂMARA MUNICIPAL,"321,106.36"
3,2026,Alhandra,1,AMORTIZAÇÃO DA DÍVIDA CONTATADA DO PODER LEGISLATIVO,"5,180.55"
4,2025,Bayeux,2001,MANUTENCAO DAS ATIVIDADES DO PODER LEGISLATIVO MUNICIPAL,"11,736,495.07"
5,2026,Bayeux,2001,MANUTENCAO DAS ATIVIDADES DO PODER LEGISLATIVO MUNICIPAL,"3,113,166.87"
6,2025,Cabedelo,2001,"Manutencao, Modernizacao e Desenvolvimento do Poder Legislativo","25,881,646.39"
7,2025,Cabedelo,2001,MANUTENCAO DAS ATIVIDADES FINS DO PODER LEGISLATIVO,"25,881,646.39"
8,2026,Cabedelo,2001,"Manutencao, Modernizacao e Desenvolvimento do Poder Legislativo","6,823,701.14"
9,2026,Cabedelo,2001,MANUTENCAO DAS ATIVIDADES FINS DO PODER LEGISLATIVO,"6,823,701.14"


#3.6. Mapeando campos ausesntes / NaN

Mapeamos campos ausentes/ valor 0/ ou com NaN , que não foram identificados na analise inicial


In [16]:
codigos_pessoal_sem_valor = codigos_pessoal_valores[
    (codigos_pessoal_valores['valor_pago'].isna()) |
    (codigos_pessoal_valores['valor_pago'] == 0)
][['ano_referencia', 'municipio', 'codigo_acao', 'acao', 'valor_pago']].reset_index(drop=True)

print(f"Total de ações sem valor OU com valor zero: {len(codigos_pessoal_sem_valor)}")

display(codigos_pessoal_sem_valor)

Total de ações sem valor OU com valor zero: 0


,ano_referencia,municipio,codigo_acao,acao,valor_pago


#3.7. Investigando Gastos Pessoais por Valor Decrescente

Nesta etapa, organizamos os gastos classificados como despesas pessoais em ordem decrescente de valor, com o objetivo de identificar registros com valores significativamente elevados.

A análise busca:

    -detectar possíveis outliers;
    identificar despesas fora do padrão esperado;
    encontrar concentrações incomuns de gastos;
    auxiliar na investigação de despesas relevantes dentro do Poder Legislativo.

In [17]:
# usa apenas códigos já filtrados pela blacklist
df_base = codigos_pessoal_valores.merge(
    codigos_pessoal_filtrado[['municipio', 'codigo_acao']],
    on=['municipio', 'codigo_acao'],
    how='inner'
)

# remove NaN, zero e nulos
codigos_pessoal_valores_real = (
    df_base[
        (df_base['valor_pago'].notna()) &
        (df_base['valor_pago'] != 0)
    ]
    .sort_values(['ano_referencia', 'valor_pago'], ascending=[True, False])
    .reset_index(drop=True)
)

# exibição
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', 200)
pd.set_option('display.float_format', '{:,.2f}'.format)

display(
    codigos_pessoal_valores_real[
        ['ano_referencia', 'municipio', 'codigo_acao', 'acao', 'valor_pago']
    ]
)

,ano_referencia,municipio,codigo_acao,acao,valor_pago
0,2025,João Pessoa,10,ENCARGOS COM PESSOAL ATIVO DA CAMARA MUNICIPAL ÁREA ADMINISTRATIVA,"90,536,271.41"
1,2025,Cabedelo,2001,"Manutencao, Modernizacao e Desenvolvimento do Poder Legislativo","25,881,646.39"
2,2025,Cabedelo,2001,"Manutencao, Modernizacao e Desenvolvimento do Poder Legislativo","25,881,646.39"
3,2025,Cabedelo,2001,MANUTENCAO DAS ATIVIDADES FINS DO PODER LEGISLATIVO,"25,881,646.39"
4,2025,Cabedelo,2001,MANUTENCAO DAS ATIVIDADES FINS DO PODER LEGISLATIVO,"25,881,646.39"
5,2025,Santa Rita,2002,MANUTENCAO DAS ATIVIDADES LEGISLATIVAS - PESSOAL/ENCARGOS SOCIAIS,"18,046,296.32"
6,2025,Santa Rita,2002,MANUTENCAO DAS ATIVIDADES LEGISLATIVAS - PESSOAL/ENCARGOS SOCIAIS,"18,046,296.32"
7,2025,Santa Rita,2002,MANUTENÇÃO DAS ATIVIDADES LEGISLATIVAS - PESSOAL/ENCARGOS SOCIAIS,"18,046,296.32"
8,2025,Santa Rita,2002,MANUTENÇÃO DAS ATIVIDADES LEGISLATIVAS - PESSOAL/ENCARGOS SOCIAIS,"18,046,296.32"
9,2025,Bayeux,2001,MANUTENCAO DAS ATIVIDADES DO PODER LEGISLATIVO MUNICIPAL,"11,736,495.07"


#4 explorando o dataset de municipios

#4.1 Explorando os Principais Campos do Dataset

Nesta primeira etapa, realizamos uma análise inicial dos principais campos presentes no dataset, buscando compreender sua estrutura e organização.

Os dados foram extraídos do sistema Instituto Brasileiro de Geografia e Estatística por meio da plataforma:

SIDRA IBGE - Tabela 9514 : https://sidra.ibge.gov.br/tabela/9514

A coleta foi realizada considerando todos os municípios da Região Metropolitana de João Pessoa, segmentados por faixa etária.

O objetivo desta etapa é:

    -compreender a estrutura dos dados;
    identificar os principais campos disponíveis;
    analisar a distribuição populacional por idade;
    preparar a base para futuras análises demográficas e comparativas.


In [18]:
import pandas as pd

url = "https://raw.githubusercontent.com/Andersonlima13/ML_TOPICOS/refs/heads/main/populacao_metropolitanas_municipio.csv"

df_pop = pd.read_csv(url)

print("Shape:", df_pop.shape)
print("\nColunas:")
print(df_pop.columns)

print("\nPreview:")
print(df_pop.head().to_string(index=False))

Shape: (986, 8)

Colunas:
Index(['codigo_municipio', 'nome_municipio', 'uf', 'ano',
       'forma_declaracao_idade', 'sexo', 'faixa_etaria', 'populacao'],
      dtype='object')

Preview:
 codigo_municipio nome_municipio uf  ano forma_declaracao_idade  sexo faixa_etaria  populacao
              NaN       Alhandra PB 2022                  Total Total      18 anos     349.00
              NaN       Alhandra PB 2022                  Total Total      19 anos     366.00
              NaN       Alhandra PB 2022                  Total Total      20 anos     376.00
              NaN       Alhandra PB 2022                  Total Total      21 anos     323.00
              NaN       Alhandra PB 2022                  Total Total      22 anos     385.00


#4.2. Aplicando Discretização nas Faixas Etárias

Nesta etapa, utilizamos a técnica de discretização para agrupar as idades da população em faixas etárias mais amplas, reduzindo a dispersão dos dados e facilitando futuras análises demográficas.

As faixas definidas foram:

18 a 29 anos — Jovem Adulto
30 a 48 anos — Adulto
49 a 59 anos — Meia Idade
60 anos ou mais — Idoso

O objetivo desse agrupamento é:

reduzir a granularidade excessiva dos dados;
facilitar análises por perfil etário;
melhorar visualizações e comparações;
permitir análises mais estratégicas sobre grupos populacionais, ao invés de idades isoladas.

In [19]:
# filtra só o total (evita dupla contagem de Homens+Mulheres)
import numpy as np

def extrair_idade(valor):
    valor = str(valor).lower().strip()

    if 'total' in valor:
        return np.nan

    if 'ou mais' in valor:
        return int(valor.split()[0])  # ex: "100 anos ou mais" → 100

    if 'ano' in valor:
        return int(valor.split()[0])  # ex: "18 anos" → 18

    return np.nan


def classificar_faixa(idade):
    if pd.isna(idade):
        return np.nan
    elif 18 <= idade <= 29:
        return 'JOVEM_ADULTO'
    elif 30 <= idade <= 48:
        return 'ADULTO'
    elif 49 <= idade <= 59:
        return 'MEIA_IDADE'
    elif idade >= 60:
        return 'IDOSO'
    else:
        return 'OUTROS'  # menores de 18


# 🔥 garantir que populacao é numérica
df_pop['populacao'] = pd.to_numeric(df_pop['populacao'], errors='coerce')

# 🔥 criar colunas necessárias
df_pop['idade_num'] = df_pop['faixa_etaria'].apply(extrair_idade)
df_pop['grupo_idade'] = df_pop['idade_num'].apply(classificar_faixa)


# 🔥 agrupamento final
df_pop_agrupado = (
    df_pop
    .dropna(subset=['grupo_idade'])  # remove inválidos
    .groupby(['nome_municipio', 'grupo_idade'])['populacao']
    .sum()
    .reset_index()
)

print("\n=== População AGRUPADA por faixa etária ===")
print(df_pop_agrupado.to_string(index=False))


=== População AGRUPADA por faixa etária ===
        nome_municipio  grupo_idade  populacao
              Alhandra       ADULTO   6,166.00
              Alhandra        IDOSO   2,400.00
              Alhandra JOVEM_ADULTO   4,170.00
              Alhandra   MEIA_IDADE   2,561.00
                Bayeux       ADULTO  23,449.00
                Bayeux        IDOSO  11,287.00
                Bayeux JOVEM_ADULTO  15,095.00
                Bayeux   MEIA_IDADE  11,853.00
               Caaporã       ADULTO   6,069.00
               Caaporã        IDOSO   2,322.00
               Caaporã JOVEM_ADULTO   3,968.00
               Caaporã   MEIA_IDADE   2,405.00
              Cabedelo       ADULTO  19,219.00
              Cabedelo        IDOSO   9,321.00
              Cabedelo JOVEM_ADULTO  11,960.00
              Cabedelo   MEIA_IDADE   8,893.00
                 Conde       ADULTO   7,807.00
                 Conde        IDOSO   3,271.00
                 Conde JOVEM_ADULTO   5,048.00
               

#5 Explorando o dataSet de renda média / receita e numero de postos formais de trabalho


#5.1 Explorando os Campos do Dataset Socioeconômico

Nesta etapa, realizamos uma análise inicial dos principais campos do dataset socioeconômico coletado através da plataforma Instituto Brasileiro de Geografia e Estatística:

Cidades IBGE --> https://cidades.ibge.gov.br/

Os dados foram coletados individualmente para cada município da Região Metropolitana de João Pessoa.

O objetivo desta análise é compreender a situação econômica e financeira dos municípios, considerando indicadores como:

    -renda média da população;
    quantidade de empregos formais;
    indicadores econômicos municipais;
    capacidade financeira proporcional de cada cidade.

A partir desses dados, buscamos investigar:

se os gastos públicos são proporcionais à capacidade econômica do município;
quais municípios possuem maior custo proporcional para manutenção do Poder Legislativo;




diferenças entre arrecadação, população economicamente ativa e despesas legislativas.
Essa etapa servirá como base para análises comparativas e construção de indicadores proporcionais entre receita, população e gastos públicos.




In [24]:
import pandas as pd


## vai ser necessario adotar alguma medida para salario, como por exemplo converter o valor para salarios
## atuais e fazer a conta pelo salario minimo
# URL do dataset
url = "https://raw.githubusercontent.com/Andersonlima13/ML_TOPICOS/refs/heads/main/rendamedia-receitas.csv"

# Leitura do arquivo
df = pd.read_csv(url)

print("=== HEAD (amostra inicial) ===")
print(df.head().to_string(index=False))

print("\n=== INFO (tipos e estrutura) ===")
df.info()

print("\n=== TIPOS DAS COLUNAS ===")
print(df.dtypes)

print("\n=== DETECÇÃO DE VALORES NULOS ===")
print(df.isnull().sum())


=== HEAD (amostra inicial) ===
     CIDADE RENDA MEDIA  ANO            valor  numero de postos formais de trabalho
Joao pessoa         2,7 2023 4.800.808.170,95                                371.34
     bayeux         1,6 2023   353.091.445,49                                 12.50
   cabedelo         2,4 2023   570.177.616,73                                 24.43
 santa rita         1,8 2023   590.335.453,08                                 24.01
   alhandra         1,9 2023      231.746.486                                  5.78

=== INFO (tipos e estrutura) ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 5 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   CIDADE                                12 non-null     object 
 1   RENDA MEDIA                           12 non-null     object 
 2   ANO                                   12 non-null     in

#5.2 Normatizando campos de cidade e valor

Nesta etapa, realizamos o processo de padronização e normalização dos campos utilizados nas análises.

As cidades tiveram suas nomenclaturas ajustadas por meio da capitalização da inicial de cada nome, trazendo maior consistência entre os datasets

Além disso, os campos numéricos foram convertidos para o tipo float, permitindo operações matemáticas e análises estatísticas de forma correta.

OS campos tratados foram:

    -valor → representa a receita do município;
    renda_media → representa a renda média da população em salários mínimos.

In [ ]:
import pandas as pd

# URL do dataset
url = "https://raw.githubusercontent.com/Andersonlima13/ML_TOPICOS/refs/heads/main/rendamedia-receitas.csv"

# Leitura do arquivo
df = pd.read_csv(url)

# Padronizar nome da cidade
df['CIDADE'] = (
    df['CIDADE']
    .astype(str)
    .str.strip()
    .str.title()
)

# Converter coluna "valor" para float
df['valor'] = (
    df['valor']
    .astype(str)
    .str.replace('R$', '', regex=False)
    .str.replace('.', '', regex=False)   # remove milhar
    .str.replace(',', '.', regex=False)  # decimal
    .str.strip()
)

df['valor'] = pd.to_numeric(df['valor'], errors='coerce')


# remove valores inválidos
df = df[df['valor'].notna()]

# ordena para  leitura
df = df.sort_values(by=['CIDADE', 'valor'], ascending=[True, False]).reset_index(drop=True)

# exibe TODOS os dados tratados
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

print("=== DATASET TRATADO (TODAS AS CIDADES) ===")
print(df.to_string(index=False))

=== DATASET TRATADO (TODAS AS CIDADES) ===
                CIDADE RENDA MEDIA  ANO            valor  numero de postos formais de trabalho
              Alhandra         1,9 2023   231,746,486.00                                  5.78
                Bayeux         1,6 2023   353,091,445.49                                 12.50
               Caapora         1,7 2023   134,826,919.00                                  5.50
              Cabedelo         2,4 2023   570,177,616.73                                 24.43
                 Conde         1,8 2023   238,729,390.00                                  7.97
Cruz Do Espirito Santo         1,6 2023    95,754,223.00                                  2.41
           Joao Pessoa         2,7 2023 4,800,808,170.95                                371.34
                Lucena         1,5 2023    80,861,738.00                                  2.48
        Pedras De Fogo         1,6 2023   190,489,681.00                                  5.97
       

#6.0 1 Investigando Dados de Cargos Políticos da Câmara

Nesta etapa, iniciamos a exploração dos dados relacionados à quantidade de cargos e posições políticas presentes nas câmaras municipais.

Primeiramente, realizamos o mapeamento dos campos disponíveis no dataset, buscando compreender:

    -a estrutura das informações;
    os tipos de cargos existentes;


Essa análise inicial permite identificar quais informações serão relevantes para futuras comparações envolvendo:

quantidade de cargos legislativos;
estrutura administrativa das câmaras;
proporcionalidade entre cargos e despesas públicas;

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/Andersonlima13/ML_TOPICOS/refs/heads/main/consulta_vagas_2024_PB.csv"

# Leitura com encoding adequado
df = pd.read_csv(url, encoding='latin-1')

print("=== CAMPOS (COLUNAS) DO DATASET ===")
print(df.columns.tolist())

print(f"\n=== SHAPE ===")
print(df.shape)

print("\n=== TIPOS DAS COLUNAS ===")
print(df.dtypes)

print("\n=== VALORES NULOS POR COLUNA ===")
print(df.isnull().sum())

print("\n=== AMOSTRA DE VALORES (para entender formato real) ===")
for col in df.columns:
    print(f"\nColuna: {col}")
    print(df[col].dropna().astype(str).unique()[:5])

=== CAMPOS (COLUNAS) DO DATASET ===
['DT_GERACAO;"HH_GERACAO";"ANO_ELEICAO";"CD_TIPO_ELEICAO";"NM_TIPO_ELEICAO";"CD_ELEICAO";"DS_ELEICAO";"DT_ELEICAO";"DT_POSSE";"SG_UF";"SG_UE";"NM_UE";"CD_CARGO";"DS_CARGO";"QT_VAGA"']

=== SHAPE ===
(671, 1)

=== TIPOS DAS COLUNAS ===
DT_GERACAO;"HH_GERACAO";"ANO_ELEICAO";"CD_TIPO_ELEICAO";"NM_TIPO_ELEICAO";"CD_ELEICAO";"DS_ELEICAO";"DT_ELEICAO";"DT_POSSE";"SG_UF";"SG_UE";"NM_UE";"CD_CARGO";"DS_CARGO";"QT_VAGA"    object
dtype: object

=== VALORES NULOS POR COLUNA ===
DT_GERACAO;"HH_GERACAO";"ANO_ELEICAO";"CD_TIPO_ELEICAO";"NM_TIPO_ELEICAO";"CD_ELEICAO";"DS_ELEICAO";"DT_ELEICAO";"DT_POSSE";"SG_UF";"SG_UE";"NM_UE";"CD_CARGO";"DS_CARGO";"QT_VAGA"    0
dtype: int64

=== AMOSTRA DE VALORES (para entender formato real) ===

Coluna: DT_GERACAO;"HH_GERACAO";"ANO_ELEICAO";"CD_TIPO_ELEICAO";"NM_TIPO_ELEICAO";"CD_ELEICAO";"DS_ELEICAO";"DT_ELEICAO";"DT_POSSE";"SG_UF";"SG_UE";"NM_UE";"CD_CARGO";"DS_CARGO";"QT_VAGA"
['08/04/2026;"16:30:17";2024;2;"Eleição Ordinár

#6.1 Isolando Dados da Região Metropolitana de João Pessoa

Nesta etapa, filtramos o dataset para considerar apenas os municípios pertencentes à Região Metropolitana de João Pessoa.

Após o isolamento dos dados, realizamos o mapeamento das posições e cargos políticos relacionados às câmaras municipais

O objetivo desta análise é:

    -identificar a estrutura legislativa de cada município;
    comparar a quantidade de cargos entre cidades;
    relacionar a estrutura política com os gastos do Poder Legislativo.




In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/Andersonlima13/ML_TOPICOS/refs/heads/main/consulta_vagas_2024_PB.csv"

# 🔥 Corrigindo leitura (separador correto)
df = pd.read_csv(url, encoding='latin-1', sep=';')

# Padronizar nomes das cidades (importante)
df['NM_UE'] = (
    df['NM_UE']
    .astype(str)
    .str.strip()
    .str.upper()
)

# Lista de cidades alvo
cidades = [
    'SANTA RITA',
    'ALHANDRA',
    'JOÃO PESSOA',
    'BAYEUX',
    'CABEDELO',
    'PITIMBU',
    'CONDE',
    'PEDRAS DE FOGO',
    'RIO TINTO',
    'LUCENA'
]

# Filtrar dataset
df_filtrado = df[df['NM_UE'].isin(cidades)]

# Agrupar por cidade, UF e cargo, somando as vagas
df_resultado_agrupado = (
    df_filtrado.groupby(['NM_UE', 'SG_UE', 'DS_CARGO'])['QT_VAGA']
    .sum()
    .reset_index()
    .sort_values(by=['NM_UE', 'DS_CARGO'])
)

# Exibir resultado
print("=== VAGAS POR CIDADE E CARGO ===")
print(df_resultado_agrupado.to_string(index=False))

=== VAGAS POR CIDADE E CARGO ===
         NM_UE  SG_UE      DS_CARGO  QT_VAGA
      ALHANDRA  19119      Prefeito        1
      ALHANDRA  19119      Vereador       11
      ALHANDRA  19119 Vice-prefeito        1
        BAYEUX  19372      Prefeito        1
        BAYEUX  19372      Vereador       17
        BAYEUX  19372 Vice-prefeito        1
      CABEDELO  19658      Prefeito        2
      CABEDELO  19658      Vereador       15
      CABEDELO  19658 Vice-prefeito        2
         CONDE  19933      Prefeito        1
         CONDE  19933      Vereador       11
         CONDE  19933 Vice-prefeito        1
   JOÃO PESSOA  20516      Prefeito        1
   JOÃO PESSOA  20516      Vereador       29
   JOÃO PESSOA  20516 Vice-prefeito        1
        LUCENA  20737      Prefeito        1
        LUCENA  20737      Vereador        9
        LUCENA  20737 Vice-prefeito        1
PEDRAS DE FOGO  21253      Prefeito        1
PEDRAS DE FOGO  21253      Vereador       11
PEDRAS DE FOGO  21253 

#6.2 Agrupando Número de Cadeiras por Cidade

Nesta etapa, realizamos a agregação dos cargos políticos por município, consolidando as informações em uma visão geral da estrutura legislativa de cada cidade.

Ao invés de separar individualmente cargos como:
prefeito;
vice-prefeito;
vereadores;


optamos por agrupar todas as posições em um único indicador relacionado à quantidade total de cadeiras e cargos da câmara municipal.

O objetivo desta abordagem é:

    -simplificar a análise dos dados;
    reduzir a granularidade das informações;
    facilitar comparações entre municípios;
    compreender o tamanho da estrutura legislativa de cada cidade de forma geral.


In [ ]:
# Agrupar ignorando o tipo de cargo
df_total_cargos = (
    df_resultado_agrupado
    .groupby(['NM_UE', 'SG_UE'])['QT_VAGA']
    .sum()
    .reset_index()
    .rename(columns={'QT_VAGA': 'TOTAL_CARGOS'})
    .sort_values(by='TOTAL_CARGOS', ascending=False)
)

print("\n=== TOTAL DE CARGOS POR CIDADE ===")
print(df_total_cargos.to_string(index=False))


=== TOTAL DE CARGOS POR CIDADE ===
         NM_UE  SG_UE  TOTAL_CARGOS
   JOÃO PESSOA  20516            31
    SANTA RITA  21750            21
      CABEDELO  19658            19
        BAYEUX  19372            19
      ALHANDRA  19119            13
         CONDE  19933            13
       PITIMBU  21393            13
PEDRAS DE FOGO  21253            13
     RIO TINTO  21598            13
        LUCENA  20737            11


## Gastos com Encargos legislativos *V2*

In [2]:
# ==========================================
# 3.2. Isolando despesas de pessoal da Câmara
# usando grupo_natureza_despesa
# ==========================================

def normalizar_texto(s):
    return (
        s.astype(str)
        .str.strip()
        .str.upper()
        .str.normalize('NFKD')
        .str.encode('ascii', errors='ignore')
        .str.decode('utf-8')
    )

# padronização
df_total['grupo_natureza_clean'] = normalizar_texto(
    df_total['grupo_natureza_despesa']
)

df_total['descricao_unidade_clean'] = normalizar_texto(
    df_total['descricao_unidade_orcamentaria']
)

# ==========================================
# FILTRO PRINCIPAL
# ==========================================
# Regras:
# 1. Câmara / Legislativo
# 2. Grupo natureza = Pessoal e Encargos Sociais
# ==========================================

codigos_pessoal = (
    df_total[
        (
            df_total['descricao_unidade_clean']
            .str.contains(r'CAMARA|LEGISLATIVO', na=False)
        )
        &
        (
            df_total['grupo_natureza_clean']
            == 'PESSOAL E ENCARGOS SOCIAIS'
        )
    ][[
        'municipio',
        'codigo_acao',
        'acao',
        'grupo_natureza_despesa'
    ]]
    .drop_duplicates()
    .sort_values(['municipio', 'codigo_acao'])
    .reset_index(drop=True)
)

# ==========================================
# EXIBIÇÃO
# ==========================================

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', 200)

display(codigos_pessoal)

NameError: name 'df_total' is not defined

## grupo_natureza_despesa
Refatorando filtro anterior

In [ ]:
# ==========================================
# 3.3. Gastos do Legislativo com
# Pessoal e Encargos Sociais
# ==========================================

# ------------------------------------------
# 1. função de normalização
# ------------------------------------------
def normalizar(s):
    return (
        s.astype(str)
        .str.strip()
        .str.upper()
        .str.normalize('NFKD')
        .str.encode('ascii', errors='ignore')
        .str.decode('utf-8')
    )


# ------------------------------------------
# 2. cópia segura
# ------------------------------------------
df_leg = df_leg.copy()


# ------------------------------------------
# 3. padronizações
# ------------------------------------------
df_leg['municipio_norm'] = normalizar(
    df_leg['municipio']
)

df_leg['grupo_natureza_norm'] = normalizar(
    df_leg['grupo_natureza_despesa']
)

df_leg['acao_norm'] = normalizar(
    df_leg['acao']
)


# ------------------------------------------
# 4. filtrar SOMENTE:
# - grupo natureza = pessoal
# ------------------------------------------
df_pessoal = (
    df_leg[
        df_leg['grupo_natureza_norm']
        == 'PESSOAL E ENCARGOS SOCIAIS'
    ]
    .copy()
)


# ------------------------------------------
# 5. agregar valores
# ------------------------------------------
codigos_pessoal_valores = (
    df_pessoal
    .groupby(
        [
            'ano_referencia',
            'municipio',
            'codigo_acao',
            'acao',
            'grupo_natureza_despesa'
        ]
    )['valor_pago']
    .sum()
    .reset_index()
)


# ------------------------------------------
# 6. remover nulos/zerados
# ------------------------------------------
codigos_pessoal_valores = (
    codigos_pessoal_valores[
        codigos_pessoal_valores['valor_pago'] > 0
    ]
)


# ------------------------------------------
# 7. ordenar pelos maiores gastos
# ------------------------------------------
codigos_pessoal_valores = (
    codigos_pessoal_valores
    .sort_values(
        [
            'valor_pago',
            'ano_referencia'
        ],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)


# ------------------------------------------
# 8. configs exibição
# ------------------------------------------
pd.set_option('display.max_colwidth', None)

pd.set_option('display.max_rows', 1000)

pd.set_option(
    'display.float_format',
    '{:,.2f}'.format
)


# ------------------------------------------
# 9. resultado final
# ------------------------------------------
display(
    codigos_pessoal_valores[
        [
            'ano_referencia',
            'municipio',
            'codigo_acao',
            'acao',
            'grupo_natureza_despesa',
            'valor_pago'
        ]
    ]
)

,ano_referencia,municipio,codigo_acao,acao,grupo_natureza_despesa,valor_pago
0,2025,JOÃO PESSOA,10,ENCARGOS COM PESSOAL ATIVO DA CAMARA MUNICIPAL ÁREA ADMINISTRATIVA,Pessoal e Encargos Sociais,7169104415
1,2026,JOÃO PESSOA,10,ENCARGOS COM PESSOAL ATIVO DA CAMARA MUNICIPAL ÁREA ADMINISTRATIVA,Pessoal e Encargos Sociais,1788838121
2,2025,CABEDELO,2001,MANUTENCAO DAS ATIVIDADES FINS DO PODER LEGISLATIVO,Pessoal e Encargos Sociais,1210299442
3,2025,JOÃO PESSOA,8,ENCARGOS COM A PREVIDENCIA NACIONAL,Pessoal e Encargos Sociais,1133375190
4,2025,BAYEUX,2001,MANUTENCAO DAS ATIVIDADES DO PODER LEGISLATIVO MUNICIPAL,Pessoal e Encargos Sociais,787267357
5,2025,SANTA RITA,2002,MANUTENÇÃO DAS ATIVIDADES LEGISLATIVAS - PESSOAL/ENCARGOS SOCIAIS,Pessoal e Encargos Sociais,473584019
6,2026,CABEDELO,2001,"Manutencao, Modernizacao e Desenvolvimento do Poder Legislativo",Pessoal e Encargos Sociais,396455310
7,2025,JOÃO PESSOA,9,ENCARGOS COM A PREVIDENCIA MUNICIPAL,Pessoal e Encargos Sociais,350529237
8,2026,JOÃO PESSOA,8,ENCARGOS COM A PREVIDENCIA NACIONAL,Pessoal e Encargos Sociais,335310011
9,2025,CONDE,2001,MANUTENÇÃO DAS ATIVIDADES DA CAMARA MUNICIPAL,Pessoal e Encargos Sociais,304227766


### REFAZENDO PASSO A PASSO 3.2 3.4 // VERIFICAR DESPESAS

In [21]:



# ------------------------------------------
# 1. função de normalização
# ------------------------------------------
def normalizar(s):
    return (
        s.astype(str)
        .str.strip()
        .str.upper()
        .str.normalize('NFKD')
        .str.encode('ascii', errors='ignore')
        .str.decode('utf-8')
    )


# ------------------------------------------
# 2. cópia segura
# ------------------------------------------
df_leg = df_leg.copy()


# ------------------------------------------
# 3. normalização
# ------------------------------------------
df_leg['grupo_natureza_norm'] = normalizar(
    df_leg['grupo_natureza_despesa']
)


# ------------------------------------------
# 4. filtro oficial
# ------------------------------------------
codigos_pessoal = (
    df_leg[
        df_leg['grupo_natureza_norm']
        == 'PESSOAL E ENCARGOS SOCIAIS'
    ][
        [
            'municipio',
            'codigo_acao',
            'acao',
            'codigo_natureza',
            'grupo_natureza_despesa'
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            'municipio',
            'codigo_acao'
        ]
    )
    .reset_index(drop=True)
)


# ------------------------------------------
# 5. exibição
# ------------------------------------------
pd.set_option('display.max_colwidth', None)

pd.set_option('display.max_rows', 500)

display(codigos_pessoal)

,municipio,codigo_acao,acao,codigo_natureza,grupo_natureza_despesa
0,Alhandra,2001,MANUTENÇÃO DAS ATIVIDADES DA CÂMARA MUNICIPAL,1,Pessoal e Encargos Sociais
1,Alhandra,2002,MANUTENÇÃO DOS ENCARGOS PREVIDENCIÁIRIOS DA CÂMARA MUNICIPAL,1,Pessoal e Encargos Sociais
2,Bayeux,2001,MANUTENCAO DAS ATIVIDADES DO PODER LEGISLATIVO MUNICIPAL,1,Pessoal e Encargos Sociais
3,Caaporã,2001,MANUTENCAO DAS ATIVIDADES CAMARA MUNICIPAL,1,Pessoal e Encargos Sociais
4,Cabedelo,2001,MANUTENCAO DAS ATIVIDADES FINS DO PODER LEGISLATIVO,1,Pessoal e Encargos Sociais
5,Cabedelo,2001,"Manutencao, Modernizacao e Desenvolvimento do Poder Legislativo",1,Pessoal e Encargos Sociais
6,Conde,2001,MANUTENÇÃO DAS ATIVIDADES DA CAMARA MUNICIPAL,1,Pessoal e Encargos Sociais
7,Conde,2001,MANUTENÇÃO DAS ATIVIDADES DA CÂMARA MUNICIPAL,1,Pessoal e Encargos Sociais
8,Cruz do Espírito Santo,2001,MANUT. SERV. LEGISLATIVOS,1,Pessoal e Encargos Sociais
9,João Pessoa,7,ENCARGOS DE EXERCICIOS ANTERIORES,1,Pessoal e Encargos Sociais


In [22]:
# ==========================================
# 3.5. Filtrando despesas do Legislativo
# com Pessoal e Encargos Sociais
# ==========================================

# Após a segmentação das despesas
# classificadas como:
#
# - Pessoal e Encargos Sociais
#
# realizamos a agregação dos valores pagos
# por município, ação e ano de referência.


# ------------------------------------------
# 1. agregar valores
# ------------------------------------------
codigos_pessoal_valores = (
    codigos_pessoal
    .merge(
        df_leg[
            [
                'municipio',
                'codigo_acao',
                'acao',
                'ano_referencia',
                'valor_pago',
                'grupo_natureza_despesa'
            ]
        ],
        on=[
            'municipio',
            'codigo_acao',
            'acao',
            'grupo_natureza_despesa'
        ],
        how='left'
    )
)


# ------------------------------------------
# 2. agregar valores reais
# ------------------------------------------
codigos_pessoal_valores = (
    codigos_pessoal_valores
    .groupby(
        [
            'ano_referencia',
            'municipio',
            'codigo_acao',
            'acao',
            'grupo_natureza_despesa'
        ]
    )['valor_pago']
    .sum()
    .reset_index()
)


# ------------------------------------------
# 3. tratar NaN
# ------------------------------------------
codigos_pessoal_valores['valor_pago'] = (
    codigos_pessoal_valores['valor_pago']
    .fillna(0)
)


# ------------------------------------------
# 4. remover zerados
# ------------------------------------------
codigos_pessoal_valores = (
    codigos_pessoal_valores[
        codigos_pessoal_valores['valor_pago'] > 0
    ]
)


# ------------------------------------------
# 5. ordenar pelos maiores gastos
# ------------------------------------------
codigos_pessoal_valores = (
    codigos_pessoal_valores
    .sort_values(
        [
            'valor_pago',
            'ano_referencia'
        ],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)


# ------------------------------------------
# 6. configs
# ------------------------------------------
pd.set_option('display.max_colwidth', None)

pd.set_option('display.max_rows', 1000)

pd.set_option(
    'display.float_format',
    '{:,.2f}'.format
)


# ------------------------------------------
# 7. resultado final
# ------------------------------------------
display(
    codigos_pessoal_valores[
        [
            'ano_referencia',
            'municipio',
            'codigo_acao',
            'acao',
            'grupo_natureza_despesa',
            'valor_pago'
        ]
    ]
)

,ano_referencia,municipio,codigo_acao,acao,grupo_natureza_despesa,valor_pago
0,2025,João Pessoa,10,ENCARGOS COM PESSOAL ATIVO DA CAMARA MUNICIPAL ÁREA ADMINISTRATIVA,Pessoal e Encargos Sociais,"90,536,271.41"
1,2026,João Pessoa,10,ENCARGOS COM PESSOAL ATIVO DA CAMARA MUNICIPAL ÁREA ADMINISTRATIVA,Pessoal e Encargos Sociais,"24,297,713.84"
2,2025,Santa Rita,2002,MANUTENÇÃO DAS ATIVIDADES LEGISLATIVAS - PESSOAL/ENCARGOS SOCIAIS,Pessoal e Encargos Sociais,"18,046,296.32"
3,2025,Cabedelo,2001,MANUTENCAO DAS ATIVIDADES FINS DO PODER LEGISLATIVO,Pessoal e Encargos Sociais,"15,878,002.57"
4,2025,João Pessoa,8,ENCARGOS COM A PREVIDENCIA NACIONAL,Pessoal e Encargos Sociais,"11,333,751.90"
5,2025,Bayeux,2001,MANUTENCAO DAS ATIVIDADES DO PODER LEGISLATIVO MUNICIPAL,Pessoal e Encargos Sociais,"8,541,638.62"
6,2025,Alhandra,2001,MANUTENÇÃO DAS ATIVIDADES DA CÂMARA MUNICIPAL,Pessoal e Encargos Sociais,"7,443,460.40"
7,2025,Conde,2001,MANUTENÇÃO DAS ATIVIDADES DA CAMARA MUNICIPAL,Pessoal e Encargos Sociais,"7,393,680.82"
8,2026,Santa Rita,2002,MANUTENCAO DAS ATIVIDADES LEGISLATIVAS - PESSOAL/ENCARGOS SOCIAIS,Pessoal e Encargos Sociais,"4,478,034.38"
9,2026,Cabedelo,2001,"Manutencao, Modernizacao e Desenvolvimento do Poder Legislativo",Pessoal e Encargos Sociais,"4,145,903.10"


In [23]:
# ==========================================
# 3.6. Mapeando campos ausentes
# ==========================================

# Identificamos registros sem valores pagos
# ou com valores zerados após o processo
# de agregação das despesas.


codigos_pessoal_sem_valor = (
    codigos_pessoal_valores[
        (
            codigos_pessoal_valores['valor_pago']
            .isna()
        )
        |
        (
            codigos_pessoal_valores['valor_pago']
            == 0
        )
    ]
    .reset_index(drop=True)
)


print(
    f"Total de registros sem valor: "
    f"{len(codigos_pessoal_sem_valor)}"
)


display(codigos_pessoal_sem_valor)

Total de registros sem valor: 0


,ano_referencia,municipio,codigo_acao,acao,grupo_natureza_despesa,valor_pago


In [ ]:
Iniciando o cruzamento de dados

In [ ]:
# ==========================================
# 4. Cruzando Receita/Renda Média
# com Gastos Legislativos
# ==========================================

# Objetivo:
# Construir uma base analítica unificada
# para investigar:
#
# - gasto legislativo proporcional
# - capacidade financeira municipal
# - impacto legislativo por receita
# - possíveis outliers proporcionais


import pandas as pd


# ==========================================
# 1. AGRUPAR GASTO LEGISLATIVO TOTAL
# POR MUNICÍPIO E ANO
# ==========================================

gastos_legislativos = (
    codigos_pessoal_valores
    .groupby(
        [
            'ano_referencia',
            'municipio'
        ]
    )['valor_pago']
    .sum()
    .reset_index()
)


# renomear para melhor leitura
gastos_legislativos = (
    gastos_legislativos
    .rename(
        columns={
            'valor_pago': 'gasto_legislativo'
        }
    )
)


# ==========================================
# 2. PADRONIZAR MUNICÍPIOS
# ==========================================

gastos_legislativos['municipio'] = (
    gastos_legislativos['municipio']
    .astype(str)
    .str.strip()
    .str.title()
)

df['CIDADE'] = (
    df['CIDADE']
    .astype(str)
    .str.strip()
    .str.title()
)


# ==========================================
# 3. CRUZAR DATASETS
# ==========================================

df_analitico = (
    gastos_legislativos
    .merge(
        df,
        left_on='municipio',
        right_on='CIDADE',
        how='left'
    )
)


# ==========================================
# 4. RENOMEAR COLUNAS
# ==========================================

df_analitico = (
    df_analitico
    .rename(
        columns={
            'valor': 'receita_municipio',
            'RENDA MEDIA': 'renda_media',
            'numero de postos formais de trabalho':
                'postos_formais'
        }
    )
)


# ==========================================
# 5. CRIAR INDICADOR PRINCIPAL
#
# gasto legislativo proporcional
# à receita municipal
# ==========================================

df_analitico['gasto_por_receita'] = (
    df_analitico['gasto_legislativo']
    / df_analitico['receita_municipio']
)


# ==========================================
# 6. CONVERTER PARA %
# ==========================================

df_analitico['gasto_por_receita_pct'] = (
    df_analitico['gasto_por_receita']
    * 100
)


# ==========================================
# 7. ORDENAR MAIORES GASTOS
# PROPORCIONAIS
# ==========================================

df_analitico = (
    df_analitico
    .sort_values(
        by='gasto_por_receita_pct',
        ascending=False
    )
    .reset_index(drop=True)
)


# ==========================================
# 8. CONFIGS VISUAIS
# ==========================================

pd.set_option('display.max_rows', None)

pd.set_option('display.max_columns', None)

pd.set_option(
    'display.float_format',
    '{:,.4f}'.format
)


# ==========================================
# 9. RESULTADO FINAL
# ==========================================

display(
    df_analitico[
        [
            'ano_referencia',
            'municipio',
            'gasto_legislativo',
            'receita_municipio',
            'gasto_por_receita_pct',
            'renda_media',
            'postos_formais'
        ]
    ]
)